In [ ]:
import random 
 
 
def load_ballots(df, rank_cols):
    return [(clean_ballot(row), 1) for row in df[rank_cols].values.tolist()]
 
 
def compute_last_place_counts(df, rank_cols):
    """
    Count how many times each candidate appears in the LAST ranking column
    (rank_cols[-1]), i.e. the 5th-choice column. Blanks (NaN) are dropped.
    """
    last_col = rank_cols[-1]
    counts = df[last_col].dropna().astype(int).value_counts()
    return counts.to_dict()
 
 
def simulate_actual_plurality_st(ballots, candidates, last_place_counts):
    cand_set = set(candidates)
    scores = {c: 0 for c in candidates}
    expanded_voters = []
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        if ranked:
            scores[ranked[0]] += weight
            for _ in range(weight):
                expanded_voters.append(ranked)
 
    # the denominator from the counts themselves
    total_last_place = sum(last_place_counts.get(c, 0) for c in candidates)
    if total_last_place > 0:
        last_place_pct = {
            c: last_place_counts.get(c, 0) / total_last_place
            for c in candidates
        }
    else:
        last_place_pct = {c: 0.0 for c in candidates}
 
    standing = set(candidates)
    elimination_order = []
    branch1_count = 0  # weighted veto (unranked candidates present)
    branch2_count = 0  # deterministic veto (fully-ranked ballot)
    unranked_size_counts = {}  # distribution of len(unranked_standing)
 
    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break
        ranked_set = set(ranked_ballot)
        unranked_standing = standing - ranked_set
 
        if unranked_standing:
            branch1_count += 1
            unranked_size_counts[len(unranked_standing)] = unranked_size_counts.get(len(unranked_standing), 0) + 1
            # Exactly one unranked-standing candidate vetoed per ballot
            weights = [last_place_pct[c] for c in unranked_standing]
            total_weight = sum(weights)
            if total_weight > 0:
                vetoed_this_pass = {
                    random.choices(list(unranked_standing), weights=weights, k=1)[0]
                }
            else:
                # none of the unranked candidates have any last-place weight
                vetoed_this_pass = {random.choice(list(unranked_standing))}
        else:
            branch2_count += 1
            # Ballot ranks every standing candidate: veto voter's lowest-ranked standing candidate
            vetoed_this_pass = set()
            for c in reversed(ranked_ballot):
                if c in standing:
                    vetoed_this_pass = {c}
                    break
 
        assert len(vetoed_this_pass) <= 1, f"Vetoed more than one candidate: {vetoed_this_pass}"
 
        newly_eliminated = []
        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                newly_eliminated.append(vetoed)
        newly_eliminated.sort(key=lambda c: (-last_place_pct[c], c))
        for c in newly_eliminated:
            if c in standing:
                standing.remove(c)
                elimination_order.append(c)
 
    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
 
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    print(f"branch1 (weighted, random) fired: {branch1_count} times")
    print(f"branch2 (deterministic) fired: {branch2_count} times")
    print(f"unranked_standing size distribution during branch1: {unranked_size_counts}")
    return winner, elimination_order
 
 
if __name__ == "__main__":
    CSV_PATH = "altered_ballots3.csv"
    RANK_COLS = ["1", "2", "3", "4", "5"]
 
   
 
    df = pd.read_csv(CSV_PATH)
    df_clean = df.dropna(subset=[RANK_COLS[0]])  # must at least have a 1st choice
 
    ballots = load_ballots(df_clean, RANK_COLS)
 
    # Derive candidates directly from the ballots (no hardcoding)
    candidates = sorted({c for ballot, _ in ballots for c in ballot})
 
    # Derive last-place counts directly from the 5th column (no hardcoding)
    last_place_counts = compute_last_place_counts(df_clean, RANK_COLS)
 
    # Sanity check: keys/types should now line up
    print("last_place_counts:", last_place_counts)
    print("candidates:", candidates)
 
    winner, elimination_order = simulate_actual_plurality_st(
        ballots, candidates, last_place_counts
    )
 